In [1]:
# first try 

In [2]:
# !pip install ultralytics easyocr opencv-python-headless

In [3]:
# from roboflow import Roboflow
# from ultralytics import YOLO

# # 1. Download Dataset from Roboflow
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# secret_value_0 = user_secrets.get_secret("ROBOFLOW_KEY")

# rf = Roboflow(api_key=secret_value_0)
# project = rf.workspace("aryans-workspace-qdqdp").project("scoreboard_dataset_final")
# dataset = project.version(1).download("yolo26")

# # 2. Train YOLOv26 Model
# model = YOLO('yolov26n.pt') 

# # Train for a quick experiment (adjust epochs for better convergence)
# results = model.train(
#     data=f"{dataset.location}/data.yaml",
#     epochs=20,
#     imgsz=640,
#     batch=8,
#     device='0,1' # Uses Kaggle GPU
# )

# # 3. Load the trained weights for inference
# trained_model = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')
# print("Model training complete and weights loaded.")

In [4]:
# import cv2
# import easyocr
# import re
# import torch
# from datetime import timedelta

# class CricketHighlightPipeline:
#     def __init__(self, yolo_model, video_path, max_duration_mins=30, sample_fps=1):
#         self.yolo_model = yolo_model
#         self.video_path = video_path
#         self.max_duration_sec = max_duration_mins * 60
#         self.sample_fps = sample_fps
#         # self.reader = easyocr.Reader(['en'], gpu=True)
#         self.reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
        
#         # Tracking state
#         self.last_valid_score = {"runs": None, "wickets": None}
#         self.highlights = []

#     def preprocess_ocr_text(self, text):
#         """Hardcode common OCR errors"""
#         replacements = {
#             'O': '0', 'o': '0', 'l': '1', 'I': '1', 'i': '1', 
#             'S': '5', 's': '5', 'B': '8', 'Z': '2', 'z': '2'
#         }
#         for k, v in replacements.items():
#             text = text.replace(k, v)
#         # Remove anything that isn't a digit, slash, or dash
#         text = re.sub(r'[^\d/-]', '', text)
#         return text

#     def extract_score(self, text):
#         """Extract runs and wickets using regex"""
#         cleaned_text = self.preprocess_ocr_text(text)
#         # Matches formats like 145/3 or 145-3
#         match = re.search(r'(\d{1,3})\s*[/|-]\s*(\d{1,2})', cleaned_text)
        
#         if match:
#             runs = int(match.group(1))
#             wickets = int(match.group(2))
#             return runs, wickets
#         return None, None

#     def validate_temporal_consistency(self, current_runs, current_wickets):
#         """Ensure the score changes are logically possible in cricket"""
#         if self.last_valid_score["runs"] is None:
#             self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}
#             return False, None

#         prev_runs = self.last_valid_score["runs"]
#         prev_wickets = self.last_valid_score["wickets"]

#         run_diff = current_runs - prev_runs
#         wicket_diff = current_wickets - prev_wickets

#         event_type = None

#         # Logical Constraints: 
#         # Runs shouldn't jump by more than 7 per ball. Wickets shouldn't jump by more than 1.
#         # Scores shouldn't go backwards (ignoring penalties/reviews for this basic pipeline).
#         if 0 <= run_diff <= 7 and 0 <= wicket_diff <= 1:
#             if run_diff == 4:
#                 event_type = "4 Runs (Boundary)"
#             elif run_diff == 6:
#                 event_type = "6 Runs (Six)"
#             elif wicket_diff == 1:
#                 event_type = "Wicket"
            
#             # Update state
#             self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}
        
#         return (event_type is not None), event_type

#     def process_video(self):
#         cap = cv2.VideoCapture(self.video_path)
#         cap = cv2.VideoCapture(self.video_path)

#         if not cap.isOpened():
#             raise ValueError(f"Error opening video: {self.video_path}")
#         fps = cap.get(cv2.CAP_PROP_FPS)
#         if fps <= 0:
#             raise ValueError("Invalid FPS from video")
        
#         frame_interval = max(1, int(fps / self.sample_fps))
        
#         frame_count = 0
        
#         while cap.isOpened():
#             ret, frame = cap.read()
#             if not ret:
#                 break
                
#             current_time_sec = frame_count / fps
            
#             # Stop if we reach the max duration (e.g., 30 mins)
#             if current_time_sec > self.max_duration_sec:
#                 break

#             # Process 1 frame per second (based on sample_fps)
#             if frame_count % frame_interval == 0:
#                 # 1. Detect Lower Third
#                 results = self.yolo_model.predict(frame, conf=0.5, verbose=False)
#                 if not results or results[0].boxes is None:
#                     continue
                
#                 for box in results[0].boxes:
#                     x1, y1, x2, y2 = map(int, box.xyxy[0])
#                     h, w = frame.shape[:2]
#                     x1, y1 = max(0, x1), max(0, y1)
#                     x2, y2 = min(w, x2), min(h, y2)
                    
#                     # 2. Crop detected region
#                     roi = frame[y1:y2, x1:x2]
                    
#                     if roi.size > 0:
#                         # Convert to grayscale for better OCR
#                         gray_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
                        
#                         # 3. Apply OCR
#                         ocr_results = self.reader.readtext(gray_roi, detail=0)
#                         combined_text = " ".join(ocr_results)
                        
#                         # 4. Extract and Validate
#                         runs, wickets = self.extract_score(combined_text)
                        
#                         if runs is not None and wickets is not None:
#                             is_highlight, event = self.validate_temporal_consistency(runs, wickets)
                            
#                             # 5. Record Highlight
#                             if is_highlight:
#                                 start_t = max(0, int(current_time_sec) - 10)
#                                 end_t = int(current_time_sec) + 5
                                
#                                 self.highlights.append({
#                                     "event": event,
#                                     "timestamp_sec": int(current_time_sec),
#                                     "interval": f"{timedelta(seconds=start_t)} to {timedelta(seconds=end_t)}",
#                                     "score": f"{runs}/{wickets}"
#                                 })
            
#             frame_count += 1

#         cap.release()
#         return self.highlights

In [5]:
# # Path to the Kaggle dataset video
# # Update this path based on where the other user's dataset is mounted in your Kaggle environment
# VIDEO_PATH = "/kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs IND.mp4" 

# print("Starting video processing pipeline...")
# pipeline = CricketHighlightPipeline(
#     yolo_model=trained_model,
#     video_path=VIDEO_PATH,
#     max_duration_mins=30,  # Only process first 30 mins
#     sample_fps=1           # 1 Frame per second for optimal performance vs accuracy
# )

# detected_highlights = pipeline.process_video()

# if not detected_highlights:
#     print("No highlights detected.")
# else:
#     for hl in detected_highlights:
#         print(f"[{hl['interval']}] Event: {hl['event']} | Score Updated to: {hl['score']}")

In [6]:
# second try 

In [7]:
# !pip install -q ultralytics easyocr opencv-python-headless tqdm

# import os
# import cv2
# import re
# import torch
# import numpy as np
# from datetime import timedelta
# from ultralytics import YOLO
# import easyocr
# from tqdm.notebook import tqdm

# # Define paths
# WEIGHTS_PATH = '/kaggle/input/datasets/loak2055/yolo-weights/best.pt'
# VIDEO_PATH = '/kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs IND.mp4'

# # GPU Selection: Utilizing both T4s
# device_yolo = "cuda:0" if torch.cuda.device_count() > 0 else "cpu"
# device_ocr = "cuda:1" if torch.cuda.device_count() > 1 else device_yolo

# print(f"YOLO running on: {device_yolo}")
# print(f"EasyOCR running on: {device_ocr}")

In [8]:
# # Initialize YOLOv8
# trained_model = YOLO(WEIGHTS_PATH)
# trained_model.to(device_yolo)

# # Initialize EasyOCR with the second GPU
# # We use the specific device index to ensure parallel utilization
# reader = easyocr.Reader(['en'], gpu=True) 
# # Note: EasyOCR handles internal torch device management, 
# # but if errors occur on T4, we fallback to default GPU logic.

# print("Models loaded and ready for inference.")

In [9]:
# class CricketHighlightPipeline:
#     def __init__(self, yolo_model, ocr_reader, video_path, max_duration_mins=None, sample_fps=1):
#         self.yolo_model = yolo_model
#         self.reader = ocr_reader
#         self.video_path = video_path
#         self.sample_fps = sample_fps
        
#         # --- FIX: Handle None for Full Match ---
#         if max_duration_mins is not None:
#             self.max_duration_sec = max_duration_mins * 60
#         else:
#             self.max_duration_sec = float('inf') # Infinity means no time limit
            
#         self.last_valid_score = {"runs": None, "wickets": None}
#         self.highlights = []

#     def preprocess_ocr_text(self, text):
#         replacements = {'O': '0', 'o': '0', 'l': '1', 'I': '1', 'i': '1', 'S': '5', 's': '5', 'B': '8', 'Z': '2', 'z': '2'}
#         for k, v in replacements.items():
#             text = text.replace(k, v)
#         return re.sub(r'[^\d/-]', '', text)

#     def extract_score(self, text):
#         cleaned_text = self.preprocess_ocr_text(text)
#         match = re.search(r'(\d{1,3})\s*[/|-]\s*(\d{1,2})', cleaned_text)
#         if match:
#             return int(match.group(1)), int(match.group(2))
#         return None, None

#     def validate_and_update(self, current_runs, current_wickets):
#         if self.last_valid_score["runs"] is None:
#             self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}
#             return False, None

#         prev_runs = self.last_valid_score["runs"]
#         prev_wickets = self.last_valid_score["wickets"]
        
#         run_diff = current_runs - prev_runs
#         wicket_diff = current_wickets - prev_wickets

#         # --- NEW: INNINGS RESET LOGIC ---
#         # If the current score is very low (e.g. < 5) and the previous score was high (> 50),
#         # treat it as a new innings and reset the baseline.
#         if current_runs < 5 and prev_runs > 50:
#             print(f"🔄 Innings break detected! Resetting baseline to {current_runs}/{current_wickets}")
#             self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}
#             return False, None

#         # Standard Cricket Logic
#         if 0 <= run_diff <= 7 and 0 <= wicket_diff <= 1:
#             event_type = None
#             if run_diff == 4: event_type = "4 Runs (Boundary)"
#             elif run_diff == 6: event_type = "6 Runs (Six)"
#             elif wicket_diff == 1: event_type = "Wicket"
            
#             if run_diff > 0 or wicket_diff > 0:
#                 self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}
            
#             return (event_type is not None), event_type
        
#         return False, None

#     def process_video(self):
#         cap = cv2.VideoCapture(self.video_path)
#         fps = cap.get(cv2.CAP_PROP_FPS)
#         total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
#         limit_frames = total_frames if self.max_duration_sec == float('inf') else int(self.max_duration_sec * fps)
#         total_to_process = min(total_frames, limit_frames)
        
#         frame_interval = max(1, int(fps / self.sample_fps))
        
#         # --- FIX: tqdm.notebook with refresh control ---
#         pbar = tqdm(total=total_to_process, desc="Analyzing Match", leave=True, mininterval=1.0)
        
#         frame_count = 0
#         while cap.isOpened():
#             # Use 'grab' for skipping to save CPU, 'read' only for the frames we need
#             if frame_count % frame_interval == 0:
#                 ret, frame = cap.read()
#             else:
#                 ret = cap.grab()
#                 frame_count += 1
#                 if frame_count % 10 == 0: # Update bar every 10 frames to keep it smooth
#                     pbar.update(10)
#                 continue

#             current_time_sec = frame_count / fps
#             if not ret or current_time_sec > self.max_duration_sec:
#                 break
                
#             # --- Inference Logic ---
#             h, w = frame.shape[:2]
#             lower_third = frame[int(h * 0.75):h, 0:w]
#             results = self.yolo_model.predict(lower_third, conf=0.6, verbose=False, device=0)
            
#             if results and len(results[0].boxes) > 0:
#                 x1, y1, x2, y2 = map(int, results[0].boxes[0].xyxy[0])
#                 roi = lower_third[y1:y2, x1:x2]
#                 if roi.size > 0:
#                     ocr_res = self.reader.readtext(cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY), detail=0)
#                     runs, wickets = self.extract_score(" ".join(ocr_res))
#                     if runs is not None:
#                         is_hl, event = self.validate_and_update(runs, wickets)
#                         if is_hl:
#                             ts = int(current_time_sec)
#                             self.highlights.append({
#                                 "event": event, "ts": ts, "score": f"{runs}/{wickets}",
#                                 "interval": f"{timedelta(seconds=max(0, ts-10))} - {timedelta(seconds=ts+5)}"
#                             })
            
#             frame_count += 1
#             pbar.update(1)

#         cap.release()
#         pbar.close() # Cleanly close the bar
#         return self.highlights

In [10]:
# # Paths (Verify these in your Kaggle sidebar)
# WEIGHTS_PATH = '/kaggle/input/datasets/loak2055/yolo-weights/best.pt'
# VIDEO_PATH = '/kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs IND.mp4'

# print("🚀 Initializing Full Match Pipeline...")

# pipeline = CricketHighlightPipeline(
#     yolo_model=trained_model,
#     ocr_reader=reader,
#     video_path=VIDEO_PATH,
#     max_duration_mins=None,  # Set to None for the whole match
#     sample_fps=1
# )

# detected_highlights = pipeline.process_video()

# print("\n" + "="*40)
# print("🏏 FULL MATCH HIGHLIGHTS SUMMARY")
# print("="*40)

# if not detected_highlights:
#     print("No highlights detected. Double-check your scoreboard detection confidence.")
# else:
#     for hl in detected_highlights:
#         print(f"[{hl['interval']}] {hl['event']} | Score: {hl['score']}")

In [11]:
# third try 

In [12]:
!pip install -q ultralytics easyocr opencv-python-headless tqdm

import os
import cv2
import re
import torch
import numpy as np
from datetime import timedelta
from ultralytics import YOLO
import easyocr
from tqdm.notebook import tqdm

WEIGHTS_PATH = '/kaggle/input/datasets/loak2055/yolo-weights/best.pt'
VIDEO_PATH = '/kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs IND.mp4'

device_yolo = "cuda:0" if torch.cuda.device_count() > 0 else "cpu"
device_ocr = "cuda:1" if torch.cuda.device_count() > 1 else device_yolo

print(f"YOLO running on: {device_yolo}")
print(f"EasyOCR running on: {device_ocr}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLO running on: cuda:0
EasyOCR running on: cuda:1


In [13]:
trained_model = YOLO(WEIGHTS_PATH)
trained_model.to(device_yolo)

reader = easyocr.Reader(['en'], gpu=True)

print("Models loaded and ready for inference.")

Models loaded and ready for inference.


In [14]:
class SceneChangeDetector:
    """Lightweight histogram-based scene-change detector.
    
    Computes an HSV colour histogram per (downscaled) frame and flags
    timestamps where consecutive histograms differ by more than `threshold`
    (chi-square distance on the normalised histogram vector).
    """

    def __init__(self, video_path, threshold=0.5, resize=(320, 240), merge_gap=0.2):
        self.video_path = video_path
        self.threshold = threshold
        self.resize = resize
        self.merge_gap = merge_gap
        self.scene_times = []

    # ---- core detection ----
    @staticmethod
    def _frame_hist(frame):
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        hist = cv2.calcHist([hsv], [0, 1], None, [50, 60], [0, 180, 0, 256])
        hist = cv2.normalize(hist, hist).flatten()
        return hist

    def detect(self):
        """Run scene-change detection over the full video. Returns sorted list of timestamps (seconds)."""
        cap = cv2.VideoCapture(self.video_path)
        if not cap.isOpened():
            print("⚠️  Cannot open video for scene detection")
            return []
        fps = cap.get(cv2.CAP_PROP_FPS)
        prev_hist = None
        frame_idx = 0
        raw_times = []

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        pbar = tqdm(total=total_frames, desc="Scene detection", leave=False, mininterval=2.0)

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            small = cv2.resize(frame, self.resize)
            hist = self._frame_hist(small)
            if prev_hist is not None:
                diff = np.linalg.norm(hist - prev_hist)
                if diff > self.threshold:
                    raw_times.append(frame_idx / fps)
            prev_hist = hist
            frame_idx += 1
            pbar.update(1)

        cap.release()
        pbar.close()

        # merge timestamps closer than merge_gap seconds
        self.scene_times = self._merge_close(raw_times)
        print(f"Scene detection: {len(self.scene_times)} boundaries found")
        return self.scene_times

    def _merge_close(self, times):
        if not times:
            return []
        merged = []
        cluster = [times[0]]
        for t in times[1:]:
            if t - cluster[-1] <= self.merge_gap:
                cluster.append(t)
            else:
                merged.append(sum(cluster) / len(cluster))
                cluster = [t]
        if cluster:
            merged.append(sum(cluster) / len(cluster))
        return merged

    # ---- boundary snapping helpers ----
    def snap_start(self, desired_start, min_stable=3.0):
        """Return the nearest scene-change >= desired_start whose NEXT scene-change
        is at least min_stable seconds later (i.e. the clip starts on a stable shot)."""
        for i, s in enumerate(self.scene_times):
            if s >= desired_start:
                if i < len(self.scene_times) - 1 and (self.scene_times[i + 1] - s) >= min_stable:
                    return s
                break
        return desired_start  # fallback: keep original

    def snap_end(self, desired_end, min_stable=3.0):
        """Return the nearest scene-change <= desired_end whose PREVIOUS scene-change
        is at least min_stable seconds earlier (i.e. the clip ends on a stable shot)."""
        for i in range(len(self.scene_times) - 1, -1, -1):
            s = self.scene_times[i]
            if s <= desired_end:
                if i > 0 and (s - self.scene_times[i - 1]) >= min_stable:
                    return s
                break
        return desired_end  # fallback: keep original


class CricketHighlightPipeline:
    def __init__(self, yolo_model, ocr_reader, video_path, max_duration_mins=None, sample_fps=1, persistence=3):
        self.yolo_model = yolo_model
        self.reader = ocr_reader
        self.video_path = video_path
        self.sample_fps = sample_fps
        self.persistence_limit = persistence
        
        if max_duration_mins is not None:
            self.max_duration_sec = max_duration_mins * 60
        else:
            self.max_duration_sec = float('inf')
            
        self.last_valid_score = {"runs": None, "wickets": None}
        self.highlights = []
        self.score_buffer = []
        self.innings_active = False

    # ---- NEW: enhanced OCR preprocessing ----
    @staticmethod
    def preprocess_roi(roi):
        """Upscale small crops + Otsu binarisation for cleaner OCR input."""
        h, w = roi.shape[:2]
        min_side = min(h, w)
        if min_side < 500:
            scale = 500 / min_side
            roi = cv2.resize(roi, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_LINEAR)
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return thresh

    def preprocess_ocr_text(self, text):
        replacements = {'O': '0', 'o': '0', 'l': '1', 'I': '1', 'i': '1', 'S': '5', 's': '5', 'B': '8', 'Z': '2', 'z': '2'}
        for k, v in replacements.items():
            text = text.replace(k, v)
        return re.sub(r'[^\d/-]', '', text)

    def extract_score(self, text):
        cleaned_text = self.preprocess_ocr_text(text)
        match = re.search(r'(\d{1,3})\s*[/|-]\s*(\d{1,2})', cleaned_text)
        if match:
            return int(match.group(1)), int(match.group(2))
        return None, None

    def get_stable_score(self, runs, wickets):
        self.score_buffer.append((runs, wickets))
        if len(self.score_buffer) > self.persistence_limit:
            self.score_buffer.pop(0)
        
        if len(self.score_buffer) == self.persistence_limit and len(set(self.score_buffer)) == 1:
            return self.score_buffer[0]
        return None, None

    def validate_and_update(self, current_runs, current_wickets):
        if current_runs == 0 and current_wickets == 0:
            is_prev_valid = self.last_valid_score["runs"] is not None and self.last_valid_score["runs"] > 0
            is_initial = self.last_valid_score["runs"] is None
            
            if (is_initial or is_prev_valid) and not self.innings_active:
                self.last_valid_score = {"runs": 0, "wickets": 0}
                self.innings_active = True
                return True, "New Innings Detected"
            elif self.innings_active:
                return False, None

        if current_runs > 0 or current_wickets > 0:
            self.innings_active = False

        if self.last_valid_score["runs"] is None:
            self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}
            return False, None

        prev_runs = self.last_valid_score["runs"]
        prev_wickets = self.last_valid_score["wickets"]
        
        run_diff = current_runs - prev_runs
        wicket_diff = current_wickets - prev_wickets

        if 0 <= run_diff <= 7 and 0 <= wicket_diff <= 1:
            event_type = None
            if run_diff == 4: event_type = "Four"
            elif run_diff == 6: event_type = "Six"
            elif wicket_diff == 1: event_type = "Wicket"
            
            if run_diff > 0 or wicket_diff > 0:
                self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}
            
            return (event_type is not None), event_type
        
        return False, None

    def process_video(self):
        cap = cv2.VideoCapture(self.video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        limit_frames = total_frames if self.max_duration_sec == float('inf') else int(self.max_duration_sec * fps)
        total_to_process = min(total_frames, limit_frames)
        
        frame_interval = max(1, int(fps / self.sample_fps))
        
        pbar = tqdm(total=total_to_process, desc="Analyzing Match", leave=True, mininterval=1.0)
        
        frame_count = 0
        while cap.isOpened():
            if frame_count % frame_interval == 0:
                ret, frame = cap.read()
            else:
                ret = cap.grab()
                frame_count += 1
                if frame_count % 10 == 0:
                    pbar.update(10)
                continue

            current_time_sec = frame_count / fps
            if not ret or current_time_sec > self.max_duration_sec:
                break
                
            h, w = frame.shape[:2]
            lower_third = frame[int(h * 0.75):h, 0:w]
            results = self.yolo_model.predict(lower_third, conf=0.45, verbose=False, device=0)
            
            if results and len(results[0].boxes) > 0:
                boxes = sorted(results[0].boxes, key=lambda x: x.conf[0].item(), reverse=True)
                x1, y1, x2, y2 = map(int, boxes[0].xyxy[0])
                roi = lower_third[y1:y2, x1:x2]
                
                if roi.size > 0:
                    # ---- CHANGED: use enhanced preprocessing instead of plain greyscale ----
                    processed_roi = self.preprocess_roi(roi)
                    ocr_res = self.reader.readtext(processed_roi, detail=0)
                    raw_runs, raw_wickets = self.extract_score(" ".join(ocr_res))
                    
                    if raw_runs is not None:
                        stable_runs, stable_wickets = self.get_stable_score(raw_runs, raw_wickets)
                        
                        if stable_runs is not None:
                            is_hl, event = self.validate_and_update(stable_runs, stable_wickets)
                            if is_hl:
                                ts = int(current_time_sec)
                                start_time = max(0, ts - 15)
                                end_time = ts + 10
                                duration = end_time - start_time
                                
                                hl_data = {
                                    "event": event,
                                    "ts": ts,
                                    "score": f"{stable_runs}/{stable_wickets}",
                                    "duration": duration,
                                    "timestamps": f"[{timedelta(seconds=start_time)}] - [{timedelta(seconds=end_time)}]"
                                }
                                
                                if event == "New Innings Detected":
                                    print(f"\n{event} | Timestamps: {hl_data['timestamps']} | Score: {hl_data['score']}")
                                self.highlights.append(hl_data)
            
            frame_count += 1
            pbar.update(1)

        cap.release()
        pbar.close()
        return self.highlights


In [15]:
WEIGHTS_PATH = '/kaggle/input/datasets/loak2055/yolo-weights/best.pt'
VIDEO_PATH = '/kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs IND.mp4'

# ---- Step 1: Run scene-change detection (one-time, lightweight) ----
print("Step 1/2: Detecting scene boundaries...")
scene_detector = SceneChangeDetector(VIDEO_PATH, threshold=0.5)
scene_times = scene_detector.detect()

# ---- Step 2: Run the OCR highlight pipeline ----
print("\nStep 2/2: Running scoreboard OCR pipeline...")
pipeline = CricketHighlightPipeline(
    yolo_model=trained_model,
    ocr_reader=reader,
    video_path=VIDEO_PATH,
    max_duration_mins=None,
    sample_fps=1,
    persistence=3 
)

detected_highlights = pipeline.process_video()

print("\n" + "="*40)
print("🏏 FULL MATCH HIGHLIGHTS SUMMARY")
print("="*40)

if not detected_highlights:
    print("No highlights detected. Double-check your scoreboard detection confidence.")
else:
    for hl in detected_highlights:
        if hl['event'] != "New Innings Detected":
            print(f"Highlight: {hl['event']} | Duration: {hl['duration']}s | Timestamps: {hl['timestamps']} | Score: {hl['score']}")


Step 1/2: Detecting scene boundaries...


Scene detection:   0%|          | 0/370914 [00:00<?, ?it/s]

Scene detection: 2210 boundaries found

Step 2/2: Running scoreboard OCR pipeline...


Analyzing Match:   0%|          | 0/370914 [00:00<?, ?it/s]


New Innings Detected | Timestamps: [0:00:00] - [0:00:11] | Score: 0/0

New Innings Detected | Timestamps: [1:40:06] - [1:40:31] | Score: 0/0

🏏 FULL MATCH HIGHLIGHTS SUMMARY
Highlight: Four | Duration: 25s | Timestamps: [0:04:25] - [0:04:50] | Score: 6/0
Highlight: Six | Duration: 25s | Timestamps: [0:07:24] - [0:07:49] | Score: 14/0
Highlight: Four | Duration: 25s | Timestamps: [0:08:36] - [0:09:01] | Score: 19/0
Highlight: Six | Duration: 25s | Timestamps: [0:12:28] - [0:12:53] | Score: 27/0
Highlight: Wicket | Duration: 25s | Timestamps: [0:15:39] - [0:16:04] | Score: 31/1
Highlight: Six | Duration: 25s | Timestamps: [0:19:56] - [0:20:21] | Score: 38/1
Highlight: Four | Duration: 25s | Timestamps: [0:22:38] - [0:23:03] | Score: 42/1
Highlight: Wicket | Duration: 25s | Timestamps: [0:34:03] - [0:34:28] | Score: 61/2
Highlight: Four | Duration: 25s | Timestamps: [0:36:15] - [0:36:40] | Score: 65/2
Highlight: Four | Duration: 25s | Timestamps: [0:38:53] - [0:39:18] | Score: 71/2
Highl

In [16]:
import os
import subprocess

# Define directories
output_dir = '/kaggle/working/results'
clips_dir = os.path.join(output_dir, 'temp_clips')
os.makedirs(clips_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'output.mp4')
concat_list_path = os.path.join(output_dir, 'concat_list.txt')

def extract_clip_robust(video_path, start, duration, clip_path):
    """
    Re-encodes the clip with frame-accurate seeking and forces audio/video sync.

    Key fixes vs. the original:
    - Removed the stream-copy (-c copy) attempt entirely.  Stream copy snaps
      the video to the nearest keyframe while leaving audio at the original
      PTS, which is the root cause of the audio delay.
    - -ss is placed BEFORE -i (input seek) for speed, combined with
      -avoid_negative_ts make_zero so that the first PTS in both streams
      is always 0 — this eliminates the offset that caused the delay.
    - -af aresample=async=1000 resamples the audio to stay locked to the
      video timeline, absorbing any residual drift between clips.
    - All clips share the same codec / container settings, so the final
      concat step can safely use -c copy without re-introducing drift.
    """
    cmd = [
        'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error',
        '-ss', str(start),           # input seek  (fast)
        '-i', video_path,
        '-t', str(duration),
        '-avoid_negative_ts', 'make_zero',   # reset PTS to 0 in every clip
        '-c:v', 'libx264', '-preset', 'ultrafast', '-crf', '23',
        '-c:a', 'aac', '-b:a', '192k',
        '-af', 'aresample=async=1000',       # fix residual audio drift
        '-movflags', '+faststart',
        clip_path
    ]
    result = subprocess.run(cmd, capture_output=True)
    if os.path.exists(clip_path) and os.path.getsize(clip_path) > 10_000:
        return True
    # Fallback: output-seek (slower but maximally accurate)
    cmd_fallback = [
        'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error',
        '-i', video_path,
        '-ss', str(start),           # output seek (slower, frame-accurate)
        '-t', str(duration),
        '-avoid_negative_ts', 'make_zero',
        '-c:v', 'libx264', '-preset', 'ultrafast', '-crf', '23',
        '-c:a', 'aac', '-b:a', '192k',
        '-af', 'aresample=async=1000',
        '-movflags', '+faststart',
        clip_path
    ]
    subprocess.run(cmd_fallback, capture_output=True)
    return os.path.exists(clip_path) and os.path.getsize(clip_path) > 10_000


if not detected_highlights:
    print("No highlights detected. Skipping video generation.")
else:
    print(f"Extracting {len(detected_highlights)} clips with scene-snapped boundaries...\n")
    valid_clips = []

    for i, hl in enumerate(detected_highlights):
        ts = hl['ts']

        raw_start = max(0, ts - 15)
        raw_end   = ts + 10

        snapped_start = scene_detector.snap_start(raw_start, min_stable=3.0)
        snapped_end   = scene_detector.snap_end(raw_end,   min_stable=3.0)

        if snapped_start >= snapped_end:
            snapped_start, snapped_end = raw_start, raw_end

        duration  = snapped_end - snapped_start
        clip_path = os.path.join(clips_dir, f"clip_{i:03d}.mp4")

        ok = extract_clip_robust(VIDEO_PATH, int(snapped_start), int(duration), clip_path)

        if ok:
            valid_clips.append(clip_path)
            delta = (
                f"(shifted {snapped_start - raw_start:+.1f}s / {snapped_end - raw_end:+.1f}s)"
                if (snapped_start != raw_start or snapped_end != raw_end)
                else "(no shift)"
            )
            print(f"✅ {hl['event']:8s} @ {ts}s  [{int(snapped_start)}s → {int(snapped_end)}s]  {delta}")
        else:
            print(f"❌ {hl['event']:8s} @ {ts}s  — extraction failed")

    # ---- Merge all clips ----
    # All clips were encoded identically, so -c copy here is safe and fast.
    print(f"\nMerging {len(valid_clips)} clips...")
    with open(concat_list_path, 'w') as f:
        for clip in valid_clips:
            f.write(f"file '{os.path.abspath(clip)}'\n")

    concat_cmd = [
        'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error',
        '-f', 'concat', '-safe', '0',
        '-i', concat_list_path,
        '-c', 'copy',          # safe: all clips share the same codec
        output_path
    ]
    subprocess.run(concat_cmd)

    if os.path.exists(output_path):
        print(f"\n🎉 SUCCESS! Final highlight video saved to: {output_path}")
    else:
        print("\n❌ Error: Failed to merge videos. Check FFmpeg logs.")

Extracting 66 clips with scene-snapped boundaries...

✅ New Innings Detected @ 1s  [0s → 10s]  (shifted +0.0s / -1.0s)
✅ Four     @ 280s  [267s → 290s]  (shifted +2.2s / +0.0s)
✅ Six      @ 459s  [444s → 469s]  (no shift)
✅ Four     @ 531s  [516s → 541s]  (no shift)
✅ Six      @ 763s  [748s → 772s]  (shifted +0.0s / -0.4s)
✅ Wicket   @ 954s  [943s → 963s]  (shifted +4.6s / -0.8s)
✅ Six      @ 1211s  [1196s → 1221s]  (no shift)
✅ Four     @ 1373s  [1358s → 1382s]  (shifted +0.4s / -0.5s)
✅ Wicket   @ 2058s  [2046s → 2063s]  (shifted +3.4s / -4.6s)
✅ Four     @ 2190s  [2176s → 2200s]  (shifted +1.3s / +0.0s)
✅ Four     @ 2348s  [2333s → 2358s]  (no shift)
✅ Six      @ 2484s  [2469s → 2494s]  (no shift)
✅ Wicket   @ 2789s  [2777s → 2799s]  (shifted +3.8s / +0.0s)
✅ Four     @ 3049s  [3034s → 3058s]  (shifted +0.0s / -0.8s)
✅ Four     @ 3239s  [3224s → 3249s]  (shifted +0.6s / +0.0s)
✅ Six      @ 3278s  [3263s → 3286s]  (shifted +0.0s / -1.1s)
✅ Four     @ 3565s  [3550s → 3575s]  (no shift

In [17]:
from IPython.display import FileLink

# Generate a direct download link for the final video
print("✅ Video ready! Click the link below to download:")
display(FileLink('results/output.mp4'))

✅ Video ready! Click the link below to download:


/kaggle/working/results/output.mp4